# Publication figures, result tables, consistency audits

These are original, unmodified Python source cells from the exported Kaggle notebook, organized by research phase. Execution outputs and counts have been cleared. Cells are in original index order within this phase. The Kaggle project workspace and some earlier phases are required; this notebook is not guaranteed to execute by itself from a clean environment. The original index for every cell is stored in cell metadata.

The historical record includes exploratory retries and some superseded checks. Do not rerun an early output-producing cell over frozen results without backing up the project first.

In [ ]:

from pathlib import Path
import hashlib
import json

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

P = Path("/kaggle/working/acoustic_linguistic_research")
TABLE = P / "results/step06_final_tables/table01_primary_models_with_ci.csv"
OUT = P / "results/step08_figures"
OUT.mkdir(parents=True, exist_ok=True)

PNG = OUT / "figure02_model_performance.png"
PDF = OUT / "figure02_model_performance.pdf"
REPORT = OUT / "figure02_model_performance_manifest.json"

EXPECTED_SHA = (
    "22512d4fdd54feec3f6658f99248653e8c9fb99e110887db64578aadfb43b4c0"
)

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

assert TABLE.is_file()
assert sha(TABLE) == EXPECTED_SHA

df = pd.read_csv(TABLE)

print("=== SOURCE TABLE COLUMNS ===")
print(df.columns.tolist())

print("\n=== SOURCE TABLE PREVIEW ===")
print(df.head(8).to_string(index=False))

# Column detection is explicit to avoid silently
# plotting the wrong metric.
def find_col(candidates):
    matches = [
        col for col in df.columns
        if col.lower() in candidates
    ]
    assert len(matches) == 1, (
        f"Expected one column from {candidates}; "
        f"found {matches}"
    )
    return matches[0]

MODEL_COL = find_col({"model", "model_name"})
MAE_COL = find_col({"mae"})
RMSE_COL = find_col({"rmse"})

data = df[[MODEL_COL, MAE_COL, RMSE_COL]].copy()
data.columns = ["model", "MAE", "RMSE"]

data["model"] = data["model"].astype(str).str.strip()

ORDER = ["M0", "A", "B1", "B2", "C1", "C2"]

assert set(data["model"]) == set(ORDER)
assert not data["model"].duplicated().any()

data = data.set_index("model").loc[ORDER].reset_index()

expected = {
    "M0": (1.3812, 1.6790),
    "A": (0.9399, 1.2066),
    "B1": (1.2426, 1.9972),
    "B2": (1.1761, 1.9518),
    "C1": (0.8972, 1.3735),
    "C2": (0.7782, 1.2461)
}

for _, row in data.iterrows():
    mae_expected, rmse_expected = expected[row["model"]]
    assert np.isclose(row["MAE"], mae_expected, atol=1e-4)
    assert np.isclose(row["RMSE"], rmse_expected, atol=1e-4)

labels = [
    "M0\nMean baseline",
    "A\nAcoustic",
    "B1\nLinguistic",
    "B2\nLinguistic + prompt",
    "C1\nFusion",
    "C2\nFusion + prompt"
]

# Two aligned panels prevent confusion between
# MAE and RMSE scales.
fig, axes = plt.subplots(
    1, 2,
    figsize=(12.5, 5.2),
    sharex=True
)

x = np.arange(len(ORDER))

for ax, metric in zip(axes, ["MAE", "RMSE"]):
    values = data[metric].to_numpy(dtype=float)

    bars = ax.bar(
        x,
        values,
        width=0.68,
        color="#778899",
        edgecolor="#30343B",
        linewidth=0.6
    )

    best_index = int(np.argmin(values))
    bars[best_index].set_color("#187B78")

    for bar, value in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            value + 0.035,
            f"{value:.3f}",
            ha="center", va="bottom",
            fontsize=9
        )

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylim(0, 2.25)
    ax.set_ylabel(metric + " (lower is better)")
    ax.set_title(metric, fontsize=13, weight="bold")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis="y", alpha=0.22)
    ax.set_axisbelow(True)

fig.suptitle(
    "Speaker-Disjoint Cross-Validated Fluency Prediction",
    fontsize=14,
    weight="bold"
)

fig.text(
    0.5, 0.015,
    "400 recordings, 20 speakers, five ASR hypotheses per recording. "
    "Fixed Ridge alpha=1.0.",
    ha="center",
    fontsize=9
)

fig.tight_layout(rect=[0, 0.055, 1, 0.94])

fig.savefig(
    PNG, dpi=300,
    bbox_inches="tight"
)
fig.savefig(
    PDF,
    bbox_inches="tight"
)
plt.show()
plt.close(fig)

assert PNG.is_file() and PNG.stat().st_size > 0
assert PDF.is_file() and PDF.stat().st_size > 0

manifest = {
    "figure": "Figure 2",
    "title": "Speaker-disjoint model performance comparison",
    "data_source": str(TABLE.relative_to(P)),
    "source_sha256": sha(TABLE),
    "model_order": ORDER,
    "ridge_alpha": 1.0,
    "original_audio_count": 400,
    "speaker_count": 20,
    "hypotheses_per_audio": 5,
    "best_mae_model": "C2",
    "best_rmse_model": "A",
    "values": data.to_dict(orient="records"),
    "png_sha256": sha(PNG),
    "pdf_sha256": sha(PDF),
    "interpretation_note": (
        "Bars show original out-of-fold point estimates. "
        "Uncertainty intervals are not shown in this figure. "
        "Prompt-aware models are secondary ablations."
    )
}

if REPORT.exists():
    assert json.loads(
        REPORT.read_text(encoding="utf-8")
    ) == manifest, "Existing figure manifest differs"
else:
    temp = REPORT.with_suffix(".tmp")
    temp.write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    temp.replace(REPORT)

print("\n=== STEP 8.10 FIGURE 2 ===")
print("Status: PASS")
print("Models:", ", ".join(ORDER))
print("Best MAE:", "C2")
print("Best RMSE:", "A")
print("Original metrics unchanged: YES")
print("PNG:", PNG)
print("PDF:", PDF)
print("Manifest:", REPORT)
print("Manifest SHA256:", sha(REPORT))


In [ ]:

from pathlib import Path
import pandas as pd
import hashlib

P = Path("/kaggle/working/acoustic_linguistic_research")

SOURCE = (
    P / "results/step06_final_tables/"
    "table01_primary_models_with_ci.csv"
)

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

print("=== STEP 8.10 FIGURE 2 SOURCE DIAGNOSTIC ===")

assert SOURCE.is_file(), f"Missing: {SOURCE}"

df = pd.read_csv(SOURCE)

print("\n=== SOURCE FILE ===")
print("Path:", SOURCE)
print("SHA256:", sha(SOURCE))
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\n=== COLUMN NAMES ===")
for col in df.columns:
    print(repr(col))

print("\n=== COMPLETE SOURCE TABLE ===")
print(df.to_string(index=False, max_colwidth=100))

print("\n=== MODEL IDENTIFIERS ===")
for col in df.columns:
    if "model" in col.lower():
        print(f"\nColumn: {col}")
        print(df[col].astype("string").unique().tolist())

print("\n=== AVAILABLE RELATED TABLES ===")
folder = P / "results/step06_final_tables"

for path in sorted(folder.glob("*.csv")):
    try:
        preview = pd.read_csv(path, nrows=2)
        print(
            path.name,
            "| columns:",
            preview.columns.tolist()[:12]
        )
    except Exception as e:
        print(path.name, "| ERROR:", str(e)[:100])

print("\n=== DIAGNOSTIC SUMMARY ===")
print("Source inspected: YES")
print("Original results modified: NO")
print("Model training: NO")
print("Figure generated: NO")


In [ ]:

from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

P = Path("/kaggle/working/acoustic_linguistic_research")
SOURCE = (
    P / "results/step06_final_tables/"
    "table01_primary_models_with_ci.csv"
)
OUT = P / "results/step08_figures"
OUT.mkdir(parents=True, exist_ok=True)

PNG = OUT / "figure02_model_performance_95ci.png"
PDF = OUT / "figure02_model_performance_95ci.pdf"
MANIFEST = OUT / "figure02_model_performance_95ci.json"

EXPECTED_SHA = (
    "22512d4fdd54feec3f6658f99248653e8c9fb99e110887db64578aadfb43b4c0"
)

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

assert SOURCE.is_file()
assert sha(SOURCE) == EXPECTED_SHA

df = pd.read_csv(SOURCE)

ORDER = ["A", "B1", "B2", "C1", "C2"]

REQUIRED = [
    "model", "MAE", "MAE_CI_low", "MAE_CI_high",
    "RMSE", "RMSE_CI_low", "RMSE_CI_high"
]

assert set(REQUIRED).issubset(df.columns)
assert set(df["model"]) == set(ORDER)
assert not df["model"].duplicated().any()

data = df.set_index("model").loc[ORDER].reset_index()

expected = {
    "A":  (0.939944, 1.206617),
    "B1": (1.242553, 1.997205),
    "B2": (1.176070, 1.951791),
    "C1": (0.897204, 1.373540),
    "C2": (0.778167, 1.246079)
}

for row in data.itertuples(index=False):
    exp_mae, exp_rmse = expected[row.model]
    assert np.isclose(row.MAE, exp_mae, atol=1e-6)
    assert np.isclose(row.RMSE, exp_rmse, atol=1e-6)

for metric in ["MAE", "RMSE"]:
    assert (
        data[f"{metric}_CI_low"] <= data[metric]
    ).all()
    assert (
        data[metric] <= data[f"{metric}_CI_high"]
    ).all()

labels = [
    "A\nAcoustic",
    "B1\nLinguistic",
    "B2\nLinguistic\n+ prompt",
    "C1\nFusion",
    "C2\nFusion\n+ prompt"
]

x = np.arange(len(ORDER))

# Colors distinguish prompt-independent and
# prompt-aware models, not statistical significance.
colors = [
    "#4D647C",
    "#4D647C",
    "#B5A081",
    "#4D647C",
    "#B5A081"
]

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

fig, axes = plt.subplots(
    1, 2,
    figsize=(12.2, 5.4),
    sharey=False
)

for ax, metric in zip(axes, ["MAE", "RMSE"]):
    values = data[metric].to_numpy(dtype=float)
    lower = data[f"{metric}_CI_low"].to_numpy(dtype=float)
    upper = data[f"{metric}_CI_high"].to_numpy(dtype=float)

    errors = np.vstack([
        values - lower,
        upper - values
    ])

    ax.bar(
        x,
        values,
        width=0.65,
        color=colors,
        edgecolor="#414141",
        linewidth=0.6,
        zorder=2
    )

    ax.errorbar(
        x,
        values,
        yerr=errors,
        fmt="none",
        ecolor="#242424",
        capsize=4,
        elinewidth=1.2,
        capthick=1.2,
        zorder=3
    )

    for i, val in enumerate(values):
        ax.text(
            i, val + 0.045,
            f"{val:.3f}",
            ha="center",
            va="bottom",
            fontsize=9
        )

    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylabel(metric + " (lower is better)")
    ax.set_title(
        metric + " with 95% bootstrap CI",
        fontsize=12,
        fontweight="bold"
    )

    ymax = max(upper) * 1.18
    ax.set_ylim(0, ymax)
    ax.grid(axis="y", alpha=0.18, zorder=0)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

from matplotlib.patches import Patch

legend_items = [
    Patch(
        facecolor="#4D647C",
        label="Prompt-independent"
    ),
    Patch(
        facecolor="#B5A081",
        label="Prompt-aware (secondary)"
    )
]

fig.legend(
    handles=legend_items,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.905),
    ncol=2,
    frameon=False,
    fontsize=9
)

fig.suptitle(
    "Fluency Prediction Performance Across Model Configurations",
    fontsize=14,
    fontweight="bold",
    y=1.005
)

fig.text(
    0.5, -0.015,
    "Five-fold speaker-disjoint CV; 400 recordings from 20 speakers; "
    "Ridge alpha=1.0. Error bars: speaker-cluster bootstrap 95% CI.",
    ha="center",
    fontsize=9
)

fig.tight_layout(rect=[0, 0.04, 1, 0.83])

fig.savefig(PNG, dpi=300, bbox_inches="tight")
fig.savefig(PDF, bbox_inches="tight")
plt.show()
plt.close(fig)

assert PNG.stat().st_size > 0
assert PDF.stat().st_size > 0

manifest = {
    "figure": "Figure 2",
    "title": "Model performance with 95% bootstrap confidence intervals",
    "source_file": str(SOURCE.relative_to(P)),
    "source_sha256": sha(SOURCE),
    "model_order": ORDER,
    "model_M0_included": False,
    "ridge_alpha": 1.0,
    "cross_validation": "5-fold speaker-disjoint",
    "confidence_interval": "speaker-cluster bootstrap 95%",
    "bootstrap_draws": 10000,
    "prompt_independent": ["A", "B1", "C1"],
    "prompt_aware_secondary": ["B2", "C2"],
    "data": data[REQUIRED].to_dict(orient="records"),
    "png_sha256": sha(PNG),
    "pdf_sha256": sha(PDF),
    "interpretation": (
        "Confidence intervals describe uncertainty in "
        "individual model metrics. Inferential claims "
        "about model differences must use the previously "
        "computed paired speaker-cluster bootstrap results, "
        "not visual overlap of these intervals."
    )
}

tmp = MANIFEST.with_suffix(".tmp")
tmp.write_text(
    json.dumps(
        manifest, indent=2, ensure_ascii=False
    ),
    encoding="utf-8"
)

if MANIFEST.exists():
    assert json.loads(
        MANIFEST.read_text(encoding="utf-8")
    ) == manifest
    tmp.unlink()
else:
    tmp.replace(MANIFEST)

print("\n=== STEP 8.10 FIGURE 2 FINAL ===")
print("Status: PASS")
print("Models:", ORDER)
print("95% bootstrap CI included: YES")
print("M0 excluded: YES")
print("Best MAE: C2")
print("Best RMSE: A")

print("\n=== OUTPUT ===")
print("PNG:", PNG)
print("PDF:", PDF)
print("Manifest:", MANIFEST)
print("Manifest SHA256:", sha(MANIFEST))
print("Original results modified: NO")


In [ ]:

from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

P = Path("/kaggle/working/acoustic_linguistic_research")
SOURCE = P / "results/step06_final_tables/table01_primary_models_with_ci.csv"
OUT = P / "results/step08_figures"
OUT.mkdir(parents=True, exist_ok=True)

PNG = OUT / "figure03_accuracy_stability_tradeoff.png"
PDF = OUT / "figure03_accuracy_stability_tradeoff.pdf"
MANIFEST = OUT / "figure03_accuracy_stability_tradeoff.json"

EXPECTED_SOURCE_SHA = (
    "22512d4fdd54feec3f6658f99248653e8c9fb99e110887db64578aadfb43b4c0"
)

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

assert SOURCE.is_file()
assert sha(SOURCE) == EXPECTED_SOURCE_SHA

df = pd.read_csv(SOURCE)
ORDER = ["A", "B1", "B2", "C1", "C2"]

required = [
    "model", "MAE", "MAE_CI_low", "MAE_CI_high",
    "mean_prediction_range", "range_CI_low", "range_CI_high"
]

assert set(required).issubset(df.columns)
assert set(df["model"]) == set(ORDER)
assert not df["model"].duplicated().any()

data = df.set_index("model").loc[ORDER].reset_index()

# Cross-check frozen headline values.
expected = {
    "A": (0.939944, 0.000000),
    "B1": (1.242553, 0.330215),
    "B2": (1.176070, 0.478216),
    "C1": (0.897204, 0.176617),
    "C2": (0.778167, 0.228505)
}

for row in data.itertuples(index=False):
    mae, rng = expected[row.model]
    assert np.isclose(row.MAE, mae, atol=1e-6)
    assert np.isclose(
        row.mean_prediction_range, rng, atol=1e-6
    )

for point, lower, upper in [
    ("MAE", "MAE_CI_low", "MAE_CI_high"),
    (
        "mean_prediction_range",
        "range_CI_low",
        "range_CI_high"
    )
]:
    assert data[lower].le(data[point]).all()
    assert data[point].le(data[upper]).all()

# Separate model classes visually.
style = {
    "A":  ("#6B7280", "s"),
    "B1": ("#2563A6", "o"),
    "B2": ("#D18C32", "D"),
    "C1": ("#2563A6", "o"),
    "C2": ("#D18C32", "D")
}

fig, ax = plt.subplots(figsize=(9.2, 6.4))

for row in data.itertuples(index=False):
    name = row.model
    color, marker = style[name]

    x = float(row.mean_prediction_range)
    y = float(row.MAE)

    xerr = np.array([
        [max(0, x - float(row.range_CI_low))],
        [max(0, float(row.range_CI_high) - x)]
    ])
    yerr = np.array([
        [max(0, y - float(row.MAE_CI_low))],
        [max(0, float(row.MAE_CI_high) - y)]
    ])

    ax.errorbar(
        x, y,
        xerr=xerr,
        yerr=yerr,
        fmt=marker,
        markersize=10,
        color=color,
        ecolor=color,
        markeredgecolor="white",
        markeredgewidth=0.9,
        elinewidth=1.2,
        capsize=4,
        alpha=0.9,
        zorder=3
    )

    offsets = {
        "A": (10, -11),
        "B1": (12, 8),
        "B2": (12, -10),
        "C1": (-29, 12),
        "C2": (12, -11)
    }

    ax.annotate(
        name,
        xy=(x, y),
        xytext=offsets[name],
        textcoords="offset points",
        fontsize=11,
        fontweight="bold",
        color=color
    )

ax.set_xlim(-0.07, 0.92)
ax.set_ylim(0.45, 1.85)

ax.set_xlabel(
    "Mean prediction range across H1–H5 "
    "(lower = more stable)",
    fontsize=11
)
ax.set_ylabel(
    "Mean absolute error (lower = more accurate)",
    fontsize=11
)

ax.set_title(
    "Accuracy–Stability Trade-off Across Model Configurations",
    fontsize=13,
    fontweight="bold",
    pad=16
)

ax.grid(alpha=0.18)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

from matplotlib.lines import Line2D

legend_items = [
    Line2D(
        [0], [0], marker="s", color="none",
        markerfacecolor="#6B7280",
        markeredgecolor="#6B7280",
        markersize=9, label="Acoustic-only (A)"
    ),
    Line2D(
        [0], [0], marker="o", color="none",
        markerfacecolor="#2563A6",
        markeredgecolor="#2563A6",
        markersize=9, label="Prompt-independent (B1, C1)"
    ),
    Line2D(
        [0], [0], marker="D", color="none",
        markerfacecolor="#D18C32",
        markeredgecolor="#D18C32",
        markersize=9, label="Prompt-aware (B2, C2)"
    )
]

ax.legend(
    handles=legend_items,
    loc="upper left",
    frameon=True,
    fontsize=9
)

fig.text(
    0.5, 0.025,
    "Error bars: marginal 95% speaker-cluster bootstrap CI "
    "(10,000 draws). Fixed Ridge alpha=1.0.",
    ha="center",
    fontsize=9
)

fig.tight_layout(rect=[0, 0.06, 1, 1])

fig.savefig(PNG, dpi=300, bbox_inches="tight")
fig.savefig(PDF, bbox_inches="tight")
plt.show()
plt.close(fig)

assert PNG.is_file() and PNG.stat().st_size > 0
assert PDF.is_file() and PDF.stat().st_size > 0

manifest = {
    "figure": "Figure 3",
    "title": "Accuracy-Stability Trade-off",
    "status": "CREATED",
    "source": str(SOURCE.relative_to(P)),
    "source_sha256": sha(SOURCE),
    "models": ORDER,
    "ridge_alpha": 1.0,
    "cross_validation": "5-fold speaker-disjoint",
    "speaker_clusters": 20,
    "bootstrap_draws": 10000,
    "x_axis": "mean_prediction_range",
    "y_axis": "MAE",
    "ci_type": "marginal speaker-cluster bootstrap 95%",
    "values": data[required].to_dict(orient="records"),
    "png_sha256": sha(PNG),
    "pdf_sha256": sha(PDF),
    "interpretation_notes": [
        "A has zero ASR-induced prediction range by design.",
        "B1 and C1 are prompt-independent.",
        "B2 and C2 are prompt-aware secondary ablations.",
        "Marginal error bars cannot replace paired model-difference CIs.",
        "The plot does not demonstrate causal effects."
    ]
}

temp = MANIFEST.with_suffix(".tmp")
temp.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

if MANIFEST.exists():
    assert json.loads(
        MANIFEST.read_text(encoding="utf-8")
    ) == manifest, "Existing manifest differs"
    temp.unlink()
else:
    temp.replace(MANIFEST)

print("\n=== STEP 8.11 FIGURE 3 ===")
print("Status: PASS")
print("Models:", ORDER)
print("Mean prediction range, C1:",
      round(expected["C1"][1], 4))
print("Mean prediction range, C2:",
      round(expected["C2"][1], 4))
print("MAE, C1:", round(expected["C1"][0], 4))
print("MAE, C2:", round(expected["C2"][0], 4))

print("\n=== SAVED FILES ===")
print("PNG:", PNG)
print("PDF:", PDF)
print("Manifest:", MANIFEST)
print("Manifest SHA256:", sha(MANIFEST))
print("Original data modified: NO")


In [ ]:

from pathlib import Path
import json, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

P = Path("/kaggle/working/acoustic_linguistic_research")
SOURCE = P / "results/step06_final_tables/step06_asr_variability_per_audio.csv"
ASSOC = P / "results/step06_final_tables/step06_variability_association.csv"
BOOT = P / "results/step06_final_tables/step06_variable_subset_bootstrap.csv"
OUT = P / "results/step08_figures"
OUT.mkdir(parents=True, exist_ok=True)

PNG = OUT / "figure04_lexical_variation_prediction_range.png"
PDF = OUT / "figure04_lexical_variation_prediction_range.pdf"
MANIFEST = OUT / "figure04_lexical_variation_prediction_range.json"

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for part in iter(lambda: f.read(1024 * 1024), b""):
            h.update(part)
    return h.hexdigest()

assert all(p.exists() for p in [SOURCE, ASSOC, BOOT])

df = pd.read_csv(
    SOURCE,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(df) == 400
assert df["audio_id"].nunique() == 400
assert df["speaker"].nunique() == 20
assert df["hypothesis_count"].eq(5).all()

MODELS = ["B1", "B2", "C1", "C2"]
assert all(f"{m}_range" in df.columns for m in MODELS)

df["lexical_unique"] = pd.to_numeric(
    df["lexical_unique"], errors="raise"
)
assert df["lexical_unique"].between(1, 5).all()

variable = df.loc[df["lexical_unique"] > 1].copy()

assert len(variable) == 172
assert variable["speaker"].nunique() == 19

EXPECTED_RHO = {
    "B1": 0.5951,
    "B2": 0.4777,
    "C1": 0.4699,
    "C2": 0.4117
}

rhos = {}

for model in MODELS:
    values = pd.to_numeric(
        variable[f"{model}_range"], errors="raise"
    )
    assert np.isfinite(values).all()
    assert values.ge(0).all()

    rho, p_naive = spearmanr(
        variable["lexical_unique"], values
    )

    assert np.isclose(
        rho, EXPECTED_RHO[model], atol=0.002
    ), f"{model} rho mismatch: {rho}"

    rhos[model] = float(rho)

# This is a descriptive visualization only.
# Speaker-cluster CIs remain in frozen STEP 6 tables;
# naive Spearman p-values are not used for inference.
fig, axes = plt.subplots(
    2, 2,
    figsize=(11.5, 8.5),
    sharex=True
)

for ax, model in zip(axes.flat, MODELS):
    ycol = f"{model}_range"

    plot_df = variable[
        ["lexical_unique", ycol]
    ].copy()

    grouped = (
        plot_df.groupby("lexical_unique")[ycol]
        .agg(["mean", "median", "count"])
        .reset_index()
    )

    # Deterministic horizontal jitter only for display.
    rng = np.random.default_rng(100 + MODELS.index(model))

    jitter = rng.uniform(
        -0.13, 0.13, len(plot_df)
    )

    ax.scatter(
        plot_df["lexical_unique"] + jitter,
        plot_df[ycol],
        s=20,
        alpha=0.38,
        color="#3C6E91",
        edgecolors="none",
        rasterized=True
    )

    # Median per lexical diversity level.
    ax.plot(
        grouped["lexical_unique"],
        grouped["median"],
        color="#B65C33",
        marker="D",
        linewidth=1.8,
        markersize=6,
        label="Group median"
    )

    # Use symmetric log display only if extreme outliers
    # otherwise hide the main distribution.
    upper = float(plot_df[ycol].max())

    if upper > 5:
        ax.set_yscale("symlog", linthresh=0.1)
        scale_label = " (symlog scale)"
    else:
        scale_label = ""

    ax.set_title(
        f"{model}  |  Spearman rho = {rhos[model]:.3f}",
        fontsize=11,
        fontweight="bold"
    )
    ax.set_ylabel(
        "Prediction range" + scale_label
    )
    ax.set_xticks([2, 3, 4, 5])
    ax.set_xlim(1.6, 5.4)
    ax.grid(alpha=0.18)
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    print(f"\n=== {model} GROUP SUMMARY ===")
    print(grouped.to_string(index=False))

for ax in axes[1, :]:
    ax.set_xlabel(
        "Unique lexical ASR hypotheses per audio (out of 5)"
    )

axes[0, 0].legend(
    loc="upper left",
    fontsize=8,
    frameon=False
)

fig.suptitle(
    "Lexical ASR Diversity and Fluency Prediction Instability",
    fontsize=14,
    fontweight="bold",
    y=0.985
)

fig.text(
    0.5, 0.012,
    "172 lexically variable recordings from 19 speakers. "
    "Spearman associations are descriptive; "
    "each point represents one recording.",
    ha="center",
    fontsize=9
)

fig.tight_layout(rect=[0, 0.04, 1, 0.95])

fig.savefig(
    PNG, dpi=300, bbox_inches="tight"
)
fig.savefig(
    PDF, bbox_inches="tight"
)
plt.show()
plt.close(fig)

assert PNG.stat().st_size > 0
assert PDF.stat().st_size > 0

manifest = {
    "figure": "Figure 4",
    "status": "CREATED",
    "title": (
        "Lexical ASR Diversity and Fluency "
        "Prediction Instability"
    ),
    "source_file": str(SOURCE.relative_to(P)),
    "source_sha256": sha(SOURCE),
    "association_source_sha256": sha(ASSOC),
    "bootstrap_source_sha256": sha(BOOT),
    "models": MODELS,
    "recording_count_total": 400,
    "lexically_variable_recordings": len(variable),
    "speaker_clusters_in_subset": int(
        variable["speaker"].nunique()
    ),
    "spearman_rho": rhos,
    "x_axis": "lexical_unique",
    "y_axis": "model-specific prediction range",
    "subset": "lexical_unique > 1",
    "interpretation_notes": [
        "Correlations are descriptive associations.",
        "Audio-level observations share speaker clusters.",
        "Naive p-values are not used for inference.",
        "Confidence intervals remain in the STEP 6 "
        "speaker-cluster bootstrap results.",
        "One extreme ASR hallucination was retained.",
        "Symlog axis is used only when maximum range exceeds 5."
    ],
    "png_sha256": sha(PNG),
    "pdf_sha256": sha(PDF)
}

temp = MANIFEST.with_suffix(".tmp")
temp.write_text(
    json.dumps(
        manifest, indent=2, ensure_ascii=False
    ),
    encoding="utf-8"
)

if MANIFEST.exists():
    assert json.loads(
        MANIFEST.read_text(encoding="utf-8")
    ) == manifest, "Manifest differs from existing version"
    temp.unlink()
else:
    temp.replace(MANIFEST)

print("\n=== STEP 8.12 FIGURE 4 ===")
print("Status: PASS")
print("Total audios:", len(df))
print("Lexically variable audios:", len(variable))
print("Speakers in subset:", variable["speaker"].nunique())

print("\nSpearman rho:")
for model, rho in rhos.items():
    print(f"  {model}: {rho:.4f}")

print("\n=== SAVED FILES ===")
print("PNG:", PNG)
print("PDF:", PDF)
print("Manifest:", MANIFEST)
print("Manifest SHA256:", sha(MANIFEST))
print("Original data modified: NO")


In [ ]:

from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

P = Path("/kaggle/working/acoustic_linguistic_research")
ASR = P / "features/asr/full/asr_hypotheses_0001-0400.csv"
DIAG = P / "results/step06_final_tables/step06_ood_diagnostic_per_hypothesis.csv"

OUT = P / "results/step08_figures"
OUT.mkdir(parents=True, exist_ok=True)

PNG = OUT / "figure05_critical_asr_hallucination.png"
PDF = OUT / "figure05_critical_asr_hallucination.pdf"
CSV = OUT / "figure05_critical_asr_hallucination_data.csv"
MANIFEST = OUT / "figure05_critical_asr_hallucination.json"

TARGET = "030210164"
ORDER = ["H1", "H2", "H3", "H4", "H5"]

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def save_immutable(path, writer):
    tmp = path.with_name(path.stem + ".temporary" + path.suffix)
    writer(tmp)
    if path.exists():
        assert sha(path) == sha(tmp), (
            f"Existing artifact differs: {path}"
        )
        tmp.unlink()
    else:
        tmp.replace(path)

assert ASR.exists() and DIAG.exists()

asr = pd.read_csv(
    ASR, dtype={"audio_id": "string", "speaker": "string"}
)
diag = pd.read_csv(
    DIAG, dtype={"audio_id": "string", "speaker": "string"}
)

required_diag = {
    "audio_id", "speaker", "hypothesis_id",
    "fluency", "c2_prediction",
    "acoustic_ridge_prediction"
}

required_asr = {
    "audio_id", "hypothesis_id", "transcription"
}

assert required_diag.issubset(diag.columns)
assert required_asr.issubset(asr.columns)

d = diag.loc[
    diag["audio_id"] == TARGET
].copy()

a = asr.loc[
    asr["audio_id"] == TARGET
].copy()

assert len(d) == 5
assert len(a) == 5
assert set(d["hypothesis_id"]) == set(ORDER)
assert set(a["hypothesis_id"]) == set(ORDER)

data = d.merge(
    a[["audio_id", "hypothesis_id", "transcription"]],
    on=["audio_id", "hypothesis_id"],
    validate="one_to_one"
)

data["word_count"] = (
    data["transcription"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

data = (
    data.set_index("hypothesis_id")
    .loc[ORDER]
    .reset_index()
)

assert data["fluency"].nunique() == 1
assert np.isclose(float(data["fluency"].iloc[0]), 7.0)

h2 = data.loc[data["hypothesis_id"] == "H2"].iloc[0]

assert int(h2["word_count"]) == 96
assert np.isclose(
    float(h2["c2_prediction"]),
    -26.1267,
    atol=0.002
)

human = float(data["fluency"].iloc[0])
acoustic = float(
    data["acoustic_ridge_prediction"].iloc[0]
)

assert np.allclose(
    data["acoustic_ridge_prediction"].to_numpy(float),
    acoustic,
    atol=1e-8
)

x = np.arange(5)
c2 = data["c2_prediction"].to_numpy(float)
words = data["word_count"].to_numpy(int)

fig, axes = plt.subplots(
    1, 2,
    figsize=(12.5, 5.4),
    gridspec_kw={"width_ratios": [1.3, 1]}
)

# Panel A: downstream scores.
ax = axes[0]

ax.plot(
    x, c2,
    marker="o",
    linewidth=2,
    markersize=8,
    color="#315E83",
    label="C2 prediction"
)

ax.scatter(
    [1], [c2[1]],
    s=155,
    marker="o",
    facecolors="none",
    edgecolors="#B44336",
    linewidths=2.2,
    zorder=5
)

ax.axhline(
    human,
    color="#333333",
    linestyle="--",
    linewidth=1.7,
    label=f"Human fluency = {human:g}"
)

ax.axhline(
    acoustic,
    color="#348369",
    linestyle=":",
    linewidth=1.8,
    label=f"Acoustic-only A = {acoustic:.2f}"
)

ax.annotate(
    f"H2: {c2[1]:.2f}",
    xy=(1, c2[1]),
    xytext=(12, 14),
    textcoords="offset points",
    color="#B44336",
    fontsize=10,
    weight="bold"
)

ax.set_xticks(x, ORDER)
ax.set_ylabel("Predicted fluency score")
ax.set_title("A. Scoring instability", fontweight="bold")
ax.grid(axis="y", alpha=0.2)
ax.legend(fontsize=8.5, loc="lower right")

# Panel B: transcript length.
ax = axes[1]

colors = [
    "#7890A3" if h != "H2" else "#B44336"
    for h in ORDER
]

bars = ax.bar(
    x, words,
    color=colors,
    edgecolor="#333333",
    linewidth=0.5
)

for bar, count in zip(bars, words):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1.5,
        str(int(count)),
        ha="center", va="bottom",
        fontsize=10
    )

ax.set_xticks(x, ORDER)
ax.set_ylabel("Transcript word count")
ax.set_title("B. ASR transcript length", fontweight="bold")
ax.set_ylim(0, max(words) * 1.16)
ax.grid(axis="y", alpha=0.2)

for ax in axes:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_axisbelow(True)

fig.suptitle(
    "Critical ASR Hallucination and Downstream Score Instability",
    fontsize=13.5,
    fontweight="bold"
)

fig.text(
    0.5, 0.015,
    f"Recording {TARGET}; observed human fluency = 7. "
    "An illustrative single-recording case, not a population estimate.",
    ha="center",
    fontsize=9
)

fig.tight_layout(rect=[0, 0.055, 1, 0.93])

# Write once; never overwrite different existing evidence.
save_immutable(
    PNG,
    lambda p: fig.savefig(
        p, dpi=300, bbox_inches="tight",
        metadata={"Software": "matplotlib"}
    )
)

save_immutable(
    PDF,
    lambda p: fig.savefig(
        p, bbox_inches="tight",
        metadata={
            "Title": "Critical ASR Hallucination Case",
            "Creator": "matplotlib",
            "CreationDate": None
        }
    )
)

plt.show()
plt.close(fig)

save_columns = [
    "audio_id", "speaker", "hypothesis_id",
    "fluency", "c2_prediction",
    "acoustic_ridge_prediction",
    "word_count", "transcription"
]

save_immutable(
    CSV,
    lambda p: data[save_columns].to_csv(
        p, index=False, encoding="utf-8"
    )
)

manifest = {
    "figure": "Figure 5",
    "status": "CREATED",
    "audio_id": TARGET,
    "speaker": str(data["speaker"].iloc[0]),
    "human_fluency": human,
    "acoustic_prediction": acoustic,
    "h2_c2_prediction": float(c2[1]),
    "h2_word_count": int(words[1]),
    "h2_absolute_error": abs(float(c2[1]) - human),
    "source_sha256": {
        str(ASR.relative_to(P)): sha(ASR),
        str(DIAG.relative_to(P)): sha(DIAG)
    },
    "png_sha256": sha(PNG),
    "pdf_sha256": sha(PDF),
    "data_csv_sha256": sha(CSV),
    "interpretation": (
        "Descriptive case study of an extreme ASR output "
        "and its association with a downstream C2 score. "
        "No score clipping or case exclusion applied."
    )
}

save_immutable(
    MANIFEST,
    lambda p: p.write_text(
        json.dumps(
            manifest, indent=2, ensure_ascii=False
        ),
        encoding="utf-8"
    )
)

print("\n=== STEP 8.13 FIGURE 5 ===")
print("Status: PASS")
print("Audio ID:", TARGET)
print("Human fluency:", human)
print("Acoustic-only prediction:", round(acoustic, 4))
print("H2 C2 prediction:", round(float(c2[1]), 4))
print("H2 absolute error:", round(abs(float(c2[1])-human), 4))

print("\n=== HYPOTHESIS DATA ===")
print(
    data[
        ["hypothesis_id", "word_count",
         "c2_prediction", "acoustic_ridge_prediction"]
    ].to_string(index=False)
)

print("\n=== OUTPUT FILES ===")
print("PNG:", PNG)
print("PDF:", PDF)
print("Data CSV:", CSV)
print("Manifest:", MANIFEST)
print("Manifest SHA256:", sha(MANIFEST))
print("Original predictions modified: NO")


In [ ]:

from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

P = Path("/kaggle/working/acoustic_linguistic_research")
SOURCE = (
    P / "results/step06_final_tables/"
    "step06_c2_critical_h2_vs_h1_contributions.csv"
)
CASE = (
    P / "results/step08_figures/"
    "figure05_critical_asr_hallucination_data.csv"
)
OUT = P / "results/step08_figures"
OUT.mkdir(parents=True, exist_ok=True)

PNG = OUT / "figure05b_h2_h1_feature_contributions.png"
PDF = OUT / "figure05b_h2_h1_feature_contributions.pdf"
CSV = OUT / "figure05b_feature_contribution_summary.csv"
REPORT = OUT / "figure05b_feature_contribution_manifest.json"

TARGET = "030210164"

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for part in iter(lambda: f.read(1024 * 1024), b""):
            h.update(part)
    return h.hexdigest()

assert SOURCE.is_file()
assert CASE.is_file()

contrib = pd.read_csv(
    SOURCE, dtype={"audio_id": "string"}
)
case = pd.read_csv(
    CASE, dtype={"audio_id": "string"}
)

required = {
    "audio_id", "feature", "feature_group",
    "prediction_delta_contribution"
}
assert required.issubset(contrib.columns)

contrib = contrib.loc[
    contrib["audio_id"] == TARGET
].copy()

case = case.loc[
    case["audio_id"] == TARGET
].copy()

assert len(contrib) > 0
assert len(case) == 5

assert set(case["hypothesis_id"]) == {
    "H1", "H2", "H3", "H4", "H5"
}

h1 = float(
    case.loc[
        case["hypothesis_id"] == "H1",
        "c2_prediction"
    ].iloc[0]
)

h2 = float(
    case.loc[
        case["hypothesis_id"] == "H2",
        "c2_prediction"
    ].iloc[0]
)

observed_delta = h2 - h1

contrib["prediction_delta_contribution"] = pd.to_numeric(
    contrib["prediction_delta_contribution"],
    errors="raise"
)

assert np.isfinite(
    contrib["prediction_delta_contribution"]
).all()

group_summary = (
    contrib.groupby("feature_group", as_index=False)
    ["prediction_delta_contribution"]
    .sum()
)

print("=== ORIGINAL FEATURE GROUP LABELS ===")
print(group_summary.to_string(index=False))

# Match documented groups without silently
# substituting unrecognized feature categories.
def categorize(name):
    value = str(name).lower().strip()
    if "acoustic" in value:
        return "Acoustic"
    if "prompt" in value:
        return "Prompt-aware"
    if "linguistic" in value or "text" in value:
        return "Linguistic"
    raise ValueError(
        f"Unrecognized feature group: {name}"
    )

contrib["plot_group"] = (
    contrib["feature_group"].map(categorize)
)

ORDER = ["Acoustic", "Linguistic", "Prompt-aware"]

summary = (
    contrib.groupby("plot_group")
    ["prediction_delta_contribution"]
    .sum()
    .reindex(ORDER, fill_value=0.0)
)

total = float(summary.sum())
residual = observed_delta - total

# Validate against the previously documented
# decomposition, allowing only rounding tolerance.
expected = {
    "Acoustic": 0.0,
    "Linguistic": -15.7110,
    "Prompt-aware": -17.7889
}

for group, value in expected.items():
    assert np.isclose(
        summary[group], value,
        rtol=0, atol=0.01
    ), f"Unexpected {group}: {summary[group]}"

assert np.isclose(
    total, observed_delta,
    rtol=0, atol=0.001
), (
    f"Contributions do not reconstruct prediction: "
    f"{total} vs {observed_delta}"
)

# Plot feature-group contributions on a common
# signed scale; no posthoc clipping.
fig, ax = plt.subplots(figsize=(9.0, 4.8))

colors = ["#738596", "#376A91", "#B36B49"]
values = summary.to_numpy(dtype=float)

bars = ax.barh(
    ORDER,
    values,
    color=colors,
    edgecolor="#363636",
    linewidth=0.6
)

for bar, val in zip(bars, values):
    ax.text(
        val - 0.35 if val < 0 else val + 0.35,
        bar.get_y() + bar.get_height()/2,
        f"{val:+.3f}",
        ha="right" if val < 0 else "left",
        va="center",
        fontsize=10
    )

ax.axvline(0, color="#333333", linewidth=1)
ax.set_xlim(min(-21, float(values.min()) * 1.17), 3)
ax.invert_yaxis()
ax.set_xlabel(
    "Contribution to C2 prediction change "
    "(H2 minus H1)"
)
ax.set_title(
    "Critical ASR Case: Feature-Group Contributions",
    fontsize=13,
    fontweight="bold",
    pad=12
)
ax.grid(axis="x", alpha=0.18)
ax.set_axisbelow(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.text(
    0.5, 0.025,
    f"Audio {TARGET}: H1 = {h1:.3f}, "
    f"H2 = {h2:.3f}; "
    f"total change = {observed_delta:+.3f}.",
    ha="center",
    fontsize=9
)

fig.tight_layout(rect=[0, 0.07, 1, 1])

fig.savefig(PNG, dpi=300, bbox_inches="tight")
fig.savefig(PDF, bbox_inches="tight")
plt.show()
plt.close(fig)

assert PNG.is_file() and PNG.stat().st_size > 0
assert PDF.is_file() and PDF.stat().st_size > 0

summary_df = pd.DataFrame({
    "feature_group": ORDER,
    "contribution_H2_minus_H1": values
})
summary_df.to_csv(CSV, index=False)

manifest = {
    "figure": "Figure 5B",
    "status": "PASS",
    "audio_id": TARGET,
    "model": "C2",
    "comparison": "H2 minus H1",
    "h1_prediction": h1,
    "h2_prediction": h2,
    "observed_prediction_delta": observed_delta,
    "summed_feature_contributions": total,
    "reconstruction_residual": residual,
    "contributions": {
        key: float(summary[key])
        for key in ORDER
    },
    "source_sha256": {
        str(SOURCE.relative_to(P)): sha(SOURCE),
        str(CASE.relative_to(P)): sha(CASE)
    },
    "png_sha256": sha(PNG),
    "pdf_sha256": sha(PDF),
    "csv_sha256": sha(CSV),
    "interpretation_note": (
        "Exact linear-model decomposition of the "
        "prediction difference between H2 and H1 "
        "within the same validation fold. "
        "This is a descriptive model attribution, "
        "not a causal estimate."
    )
}

tmp = REPORT.with_suffix(".tmp")
tmp.write_text(
    json.dumps(
        manifest, indent=2, ensure_ascii=False
    ),
    encoding="utf-8"
)

if REPORT.exists():
    assert json.loads(
        REPORT.read_text(encoding="utf-8")
    ) == manifest, "Existing report differs"
    tmp.unlink()
else:
    tmp.replace(REPORT)

print("\n=== STEP 8.14 FIGURE 5B ===")
print("Status: PASS")
print("H1 prediction:", round(h1, 6))
print("H2 prediction:", round(h2, 6))
print("Observed H2-H1:", round(observed_delta, 6))

print("\n=== FEATURE CONTRIBUTIONS ===")
print(summary_df.to_string(index=False))
print("Summed contribution:", round(total, 6))
print("Reconstruction residual:", round(residual, 10))

print("\n=== OUTPUT FILES ===")
print("PNG:", PNG)
print("PDF:", PDF)
print("CSV:", CSV)
print("Manifest:", REPORT)
print("Manifest SHA256:", sha(REPORT))
print("Original predictions modified: NO")


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib

P = Path("/kaggle/working/acoustic_linguistic_research")
SRC = P / "results/step06_final_tables"
OUT = P / "results/step08_publication_tables"
OUT.mkdir(parents=True, exist_ok=True)

MODELS = SRC / "table01_primary_models_with_ci.csv"
PAIRED = SRC / "table02_paired_differences_with_ci.csv"

OUT_MODELS = OUT / "table02a_model_performance.csv"
OUT_PAIRED = OUT / "table02b_paired_bootstrap_comparisons.csv"
MANIFEST = OUT / "step08_15_publication_tables_manifest.json"

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

assert MODELS.exists() and PAIRED.exists()

model_df = pd.read_csv(MODELS)
paired_df = pd.read_csv(PAIRED)

required_models = {
    "model", "MAE", "MAE_CI_low", "MAE_CI_high",
    "RMSE", "RMSE_CI_low", "RMSE_CI_high",
    "mean_prediction_range", "range_CI_low",
    "range_CI_high"
}

required_paired = {
    "comparison", "metric", "difference",
    "CI_low", "CI_high", "CI_excludes_zero"
}

assert required_models.issubset(model_df.columns)
assert required_paired.issubset(paired_df.columns)

ORDER = ["A", "B1", "B2", "C1", "C2"]
assert set(model_df["model"]) == set(ORDER)
assert not model_df["model"].duplicated().any()

model_df = (
    model_df.set_index("model")
    .loc[ORDER]
    .reset_index()
)

def fmt_ci(point, low, high):
    return f"{point:.4f} [{low:.4f}, {high:.4f}]"

publication_models = pd.DataFrame({
    "Model": model_df["model"],
    "MAE (95% CI)": [
        fmt_ci(r.MAE, r.MAE_CI_low, r.MAE_CI_high)
        for r in model_df.itertuples(index=False)
    ],
    "RMSE (95% CI)": [
        fmt_ci(r.RMSE, r.RMSE_CI_low, r.RMSE_CI_high)
        for r in model_df.itertuples(index=False)
    ],
    "Prediction range (95% CI)": [
        fmt_ci(
            r.mean_prediction_range,
            r.range_CI_low,
            r.range_CI_high
        )
        for r in model_df.itertuples(index=False)
    ],
    "Model role": [
        "Primary: acoustic-only",
        "Primary: linguistic",
        "Secondary: linguistic + prompt",
        "Primary: acoustic + linguistic",
        "Secondary: full fusion + prompt"
    ]
})

# Keep source comparison and metric strings unchanged.
# Do not relabel or reconstruct any missing comparisons.
paired_df["difference"] = pd.to_numeric(
    paired_df["difference"], errors="raise"
)
paired_df["CI_low"] = pd.to_numeric(
    paired_df["CI_low"], errors="raise"
)
paired_df["CI_high"] = pd.to_numeric(
    paired_df["CI_high"], errors="raise"
)

assert np.isfinite(
    paired_df[["difference", "CI_low", "CI_high"]]
).all().all()

assert (
    paired_df["CI_low"] <= paired_df["difference"]
).all()

assert (
    paired_df["difference"] <= paired_df["CI_high"]
).all()

calculated_excludes_zero = (
    (paired_df["CI_low"] > 0)
    | (paired_df["CI_high"] < 0)
)

source_flag = (
    paired_df["CI_excludes_zero"]
    .astype(str).str.strip().str.lower()
)

assert source_flag.isin(["true", "false"]).all()
assert (
    source_flag.eq("true").to_numpy()
    == calculated_excludes_zero.to_numpy()
).all()

publication_paired = paired_df.copy()
publication_paired["Difference (95% CI)"] = [
    fmt_ci(r.difference, r.CI_low, r.CI_high)
    for r in paired_df.itertuples(index=False)
]

publication_paired["CI excludes zero"] = (
    np.where(calculated_excludes_zero, "Yes", "No")
)

publication_paired = publication_paired[
    ["comparison", "metric",
     "Difference (95% CI)", "CI excludes zero"]
].rename(columns={
    "comparison": "Comparison",
    "metric": "Metric"
})

# Verify that previously reported key contrasts are present,
# regardless of ordering or cosmetic spaces in the source.
def norm(value):
    return (
        str(value).lower()
        .replace(" ", "")
        .replace("−", "-")
        .replace("–", "-")
        .replace("_", "")
    )

def find_contrast(comparison, metric_tokens):
    rows = paired_df[
        paired_df["comparison"].map(norm).eq(norm(comparison))
        & paired_df["metric"].map(
            lambda s: any(
                token in norm(s) for token in metric_tokens
            )
        )
    ]
    return rows

checks = [
    ("C2-A", ["mae"], -0.1618),
    ("C2-C1", ["mae"], -0.1190),
    ("C2-C1", ["range"], +0.0519)
]

verified = []
for comparison, tokens, expected in checks:
    matches = find_contrast(comparison, tokens)
    if len(matches) == 1:
        value = float(matches.iloc[0]["difference"])
        assert np.isclose(value, expected, atol=0.0002)
        verified.append(comparison + " / " + tokens[0])
    else:
        print(
            "Contrast not uniquely identified by name:",
            comparison, tokens,
            "matches:", len(matches)
        )

def save_immutable(frame, path):
    temp = path.with_suffix(".tmp")
    frame.to_csv(temp, index=False, encoding="utf-8")
    if path.exists():
        assert sha(path) == sha(temp), (
            f"Existing CSV differs: {path}"
        )
        temp.unlink()
    else:
        temp.replace(path)

save_immutable(publication_models, OUT_MODELS)
save_immutable(publication_paired, OUT_PAIRED)

manifest = {
    "step": "8.15",
    "status": "PASS",
    "models": ORDER,
    "model_count": len(publication_models),
    "paired_comparison_rows": len(publication_paired),
    "key_contrasts_identified": verified,
    "source_hashes": {
        str(MODELS.relative_to(P)): sha(MODELS),
        str(PAIRED.relative_to(P)): sha(PAIRED)
    },
    "output_hashes": {
        OUT_MODELS.name: sha(OUT_MODELS),
        OUT_PAIRED.name: sha(OUT_PAIRED)
    },
    "statistical_notes": [
        "95% CIs use speaker-cluster bootstrap.",
        "Paired contrasts use speaker-level paired resampling.",
        "Multiple-comparison adjustments were not applied.",
        "Prompt-aware B2 and C2 are secondary ablations.",
        "A is ASR-invariant by design.",
        "No model retraining or new statistical tests."
    ]
}

temp = MANIFEST.with_suffix(".tmp")
temp.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

if MANIFEST.exists():
    assert json.loads(
        MANIFEST.read_text(encoding="utf-8")
    ) == manifest
    temp.unlink()
else:
    temp.replace(MANIFEST)

print("=== STEP 8.15 PUBLICATION TABLES ===")
print("Status: PASS")
print("Model rows:", len(publication_models))
print("Paired comparison rows:", len(publication_paired))
print("Verified key contrasts:", verified)

print("\n=== TABLE 2A — MODEL PERFORMANCE ===")
print(publication_models.to_string(index=False))

print("\n=== TABLE 2B — PAIRED COMPARISONS ===")
print(publication_paired.to_string(index=False))

print("\n=== SAVED FILES ===")
print(OUT_MODELS)
print(OUT_PAIRED)
print(MANIFEST)
print("Manifest SHA256:", sha(MANIFEST))
print("Original frozen results modified: NO")


In [ ]:

from pathlib import Path
import json
import hashlib
import re

import numpy as np
import pandas as pd

P = Path("/kaggle/working/acoustic_linguistic_research")
SRC = P / "results/step06_final_tables"
OUT = P / "results/step08_publication_tables"
OUT.mkdir(parents=True, exist_ok=True)

SOURCE = SRC / "table02_paired_differences_with_ci.csv"
EXISTING = OUT / "table02b_paired_bootstrap_comparisons.csv"

CORRECTED = OUT / "step08_15a_verified_key_contrasts.csv"
REPORT = OUT / "step08_15a_contrast_verification.json"

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

assert SOURCE.is_file()
assert EXISTING.is_file()

df = pd.read_csv(SOURCE)
pub = pd.read_csv(EXISTING)

assert len(df) == 18
assert len(pub) == 18

required = {
    "comparison", "metric", "difference",
    "CI_low", "CI_high", "CI_excludes_zero"
}
assert required.issubset(df.columns)

# Normalize historical source labels such as
# C2_minus_C1, C2-C1 and C2 minus C1 consistently.
def normalize_contrast(value):
    s = str(value).lower().strip()
    s = s.replace("−", "-").replace("–", "-")
    s = re.sub(r"\s*[_ -]*minus[_ -]*\s*", "-", s)
    s = re.sub(r"[\s_]+", "", s)
    return s

def normalize_metric(value):
    s = str(value).lower().strip()
    s = re.sub(r"[\s_-]+", "", s)
    if s in ("meanrange", "meanpredictionrange", "range"):
        return "range"
    return s

df["canonical_comparison"] = (
    df["comparison"].map(normalize_contrast)
)
df["canonical_metric"] = df["metric"].map(normalize_metric)

assert not df.duplicated(
    ["canonical_comparison", "canonical_metric"]
).any()

targets = [
    ("C2-A", "mae", -0.1618, -0.2367, -0.0858),
    ("C2-C1", "mae", -0.1190, -0.1841, -0.0598),
    ("C2-C1", "rmse", -0.1275, -0.1885, -0.0710),
    ("C2-C1", "range", +0.0519, +0.0095, +0.1013),
]

verified = []

for comp, metric, value, lo, hi in targets:
    match = df.loc[
        df["canonical_comparison"].eq(
            normalize_contrast(comp)
        )
        & df["canonical_metric"].eq(metric)
    ]

    assert len(match) == 1, (
        f"Contrast not unique: {comp}/{metric}; "
        f"matches={len(match)}"
    )

    row = match.iloc[0]

    assert np.isclose(
        row["difference"], value,
        rtol=0, atol=0.00015
    )
    assert np.isclose(
        row["CI_low"], lo,
        rtol=0, atol=0.00015
    )
    assert np.isclose(
        row["CI_high"], hi,
        rtol=0, atol=0.00015
    )

    excludes_zero = bool(
        (row["CI_low"] > 0)
        or (row["CI_high"] < 0)
    )
    assert excludes_zero

    verified.append({
        "comparison": comp,
        "metric": metric,
        "source_comparison": row["comparison"],
        "source_metric": row["metric"],
        "difference": float(row["difference"]),
        "ci_low": float(row["CI_low"]),
        "ci_high": float(row["CI_high"]),
        "nominal_ci_excludes_zero": excludes_zero
    })

result = pd.DataFrame(verified)

def write_csv_immutable(frame, path):
    temp = path.with_suffix(".tmp")
    frame.to_csv(temp, index=False)
    if path.exists():
        assert sha(path) == sha(temp), (
            f"Existing output differs: {path}"
        )
        temp.unlink()
    else:
        temp.replace(path)

write_csv_immutable(result, CORRECTED)

report = {
    "step": "8.15a",
    "status": "PASS",
    "correction": (
        "Contrast names in source use _minus_ instead "
        "of hyphen notation; original 8.15 matching "
        "logic failed to identify them."
    ),
    "source_comparison_rows": len(df),
    "publication_comparison_rows": len(pub),
    "key_contrasts_expected": len(targets),
    "key_contrasts_verified": len(verified),
    "multiple_comparison_adjustment": False,
    "model_retraining": False,
    "bootstrap_rerun": False,
    "original_tables_modified": False,
    "source_sha256": sha(SOURCE),
    "existing_publication_table_sha256": sha(EXISTING),
    "verified_contrasts_sha256": sha(CORRECTED)
}

temp = REPORT.with_suffix(".tmp")
temp.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

if REPORT.exists():
    assert json.loads(
        REPORT.read_text(encoding="utf-8")
    ) == report
    temp.unlink()
else:
    temp.replace(REPORT)

print("=== STEP 8.15A CONTRAST VERIFICATION ===")
print("Status:", report["status"])
print("Total source contrasts:", len(df))
print("Key contrasts verified:", len(verified), "/", len(targets))

print("\n=== VERIFIED CONTRASTS ===")
print(result.to_string(index=False))

print("\n=== SOURCE PRESERVATION ===")
print("Original STEP 6 table modified: NO")
print("Original STEP 8.15 publication table modified: NO")
print("Bootstrap rerun: NO")

print("\n=== SAVED FILES ===")
print("Corrected contrast CSV:", CORRECTED)
print("Verification report:", REPORT)
print("Report SHA256:", sha(REPORT))


In [ ]:

from pathlib import Path
import json
import hashlib
import pandas as pd
import numpy as np

P = Path("/kaggle/working/acoustic_linguistic_research")
SRC = P / "results/step06_final_tables"
OUT = P / "results/step08_publication_tables"
OUT.mkdir(parents=True, exist_ok=True)

SOURCE = SRC / "table03_anomaly_sensitivity.csv"
PAIRED = SRC / "table04_sensitivity_paired_differences.csv"
MAIN = SRC / "table01_primary_models_with_ci.csv"

TABLE = OUT / "table03_anomaly_sensitivity_publication.csv"
REPORT = OUT / "step08_16_anomaly_sensitivity_manifest.json"

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

for path in [SOURCE, PAIRED, MAIN]:
    assert path.is_file(), f"Missing source: {path}"

data = pd.read_csv(SOURCE)
paired = pd.read_csv(PAIRED)
main = pd.read_csv(MAIN)

required = [
    "condition", "model", "audio_count",
    "hypothesis_count", "speaker_count",
    "MAE", "RMSE", "mean_prediction_range",
    "median_prediction_range", "p95_prediction_range"
]
assert set(required).issubset(data.columns)

assert not data.duplicated(["condition", "model"]).any()

for col in ["audio_count", "hypothesis_count", "speaker_count"]:
    data[col] = pd.to_numeric(data[col], errors="raise")
    assert data[col].notna().all()
    assert (data[col] > 0).all()
    assert (data[col] % 1 == 0).all()

for col in [
    "MAE", "RMSE", "mean_prediction_range",
    "median_prediction_range", "p95_prediction_range"
]:
    data[col] = pd.to_numeric(data[col], errors="raise")
    assert np.isfinite(data[col]).all()
    assert data[col].ge(0).all()

assert data["audio_count"].le(400).all()
assert data["speaker_count"].le(20).all()

# Five ASR hypotheses per recording are retained
# within each reported sensitivity condition.
assert (
    data["hypothesis_count"]
    == 5 * data["audio_count"]
).all()

# Verify that the original full sample can be
# identified by counts rather than guessed labels.
full = data.loc[
    (data["audio_count"] == 400)
    & (data["hypothesis_count"] == 2000)
    & (data["speaker_count"] == 20)
]

assert not full.empty, "No full-sample condition identified"

full_conditions = full["condition"].unique().tolist()
assert len(full_conditions) == 1, (
    f"Multiple full-sample conditions: {full_conditions}"
)

full_label = full_conditions[0]
assert set(["A", "B1", "B2", "C1", "C2"]).issubset(
    set(main["model"])
)

# Match full-sample sensitivity rows against
# frozen original metrics wherever present.
checked_models = []

for row in full.itertuples(index=False):
    matching = main.loc[main["model"] == row.model]

    if len(matching) != 1:
        continue

    frozen = matching.iloc[0]

    assert np.isclose(
        row.MAE, float(frozen["MAE"]),
        rtol=0, atol=1e-6
    ), f"MAE mismatch for {row.model}"

    assert np.isclose(
        row.RMSE, float(frozen["RMSE"]),
        rtol=0, atol=1e-6
    ), f"RMSE mismatch for {row.model}"

    checked_models.append(row.model)

# Keep original condition names and exact source
# point estimates, only format publication numbers.
publication = data.copy()

publication["MAE"] = publication["MAE"].round(4)
publication["RMSE"] = publication["RMSE"].round(4)
publication["mean_prediction_range"] = (
    publication["mean_prediction_range"].round(4)
)
publication["median_prediction_range"] = (
    publication["median_prediction_range"].round(4)
)
publication["p95_prediction_range"] = (
    publication["p95_prediction_range"].round(4)
)

condition_order = data["condition"].drop_duplicates().tolist()
model_order = ["A", "B1", "B2", "C1", "C2"]

publication["condition"] = pd.Categorical(
    publication["condition"],
    categories=condition_order,
    ordered=True
)
publication["model"] = pd.Categorical(
    publication["model"],
    categories=model_order,
    ordered=True
)

# If other model IDs exist, avoid dropping them.
extra_models = sorted(
    set(data["model"]) - set(model_order)
)
if extra_models:
    publication["model"] = pd.Categorical(
        data["model"],
        categories=model_order + extra_models,
        ordered=True
    )

publication = publication.sort_values(
    ["condition", "model"]
).reset_index(drop=True)

publication["condition"] = publication["condition"].astype(str)
publication["model"] = publication["model"].astype(str)

def safe_csv(frame, path):
    temp = path.with_suffix(".tmp")
    frame.to_csv(temp, index=False, encoding="utf-8")
    if path.exists():
        assert sha(path) == sha(temp), (
            f"Existing publication table differs: {path}"
        )
        temp.unlink()
    else:
        temp.replace(path)

safe_csv(publication, TABLE)

# Validate paired-sensitivity source structure,
# without generating new inferential intervals.
assert set([
    "condition", "comparison", "MAE_difference",
    "mean_range_difference"
]).issubset(paired.columns)

report = {
    "step": "8.16",
    "status": "PASS",
    "publication_rows": len(publication),
    "conditions": condition_order,
    "models": sorted(data["model"].unique().tolist()),
    "full_sample_condition": full_label,
    "full_sample_models_checked": checked_models,
    "source_hashes": {
        str(path.relative_to(P)): sha(path)
        for path in [SOURCE, PAIRED, MAIN]
    },
    "table_sha256": sha(TABLE),
    "original_results_modified": False,
    "models_retrained": False,
    "exclusions_posthoc": True,
    "interpretation_note": (
        "The full 400-recording dataset remains the "
        "primary analysis. Exclusions are post-hoc "
        "sensitivity checks and are not independent "
        "confirmatory analyses."
    )
}

temp = REPORT.with_suffix(".tmp")
temp.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

if REPORT.exists():
    assert json.loads(
        REPORT.read_text(encoding="utf-8")
    ) == report, "Existing manifest differs"
    temp.unlink()
else:
    temp.replace(REPORT)

print("=== STEP 8.16 ANOMALY SENSITIVITY TABLE ===")
print("Status:", report["status"])
print("Conditions:", condition_order)
print("Publication rows:", len(publication))
print("Full sample condition:", full_label)
print("Frozen models cross-checked:", checked_models)

print("\n=== TABLE 3 — ANOMALY SENSITIVITY ===")
print(publication.to_string(index=False))

print("\n=== EXISTING PAIRED SENSITIVITY RESULTS ===")
print(paired.to_string(index=False))

print("\n=== SAVED FILES ===")
print("Publication CSV:", TABLE)
print("Manifest:", REPORT)
print("Manifest SHA256:", sha(REPORT))
print("Original frozen data modified: NO")
print("Bootstrap rerun: NO")


In [ ]:

from pathlib import Path
import json
import hashlib
import pandas as pd

P = Path("/kaggle/working/acoustic_linguistic_research")
OUT = P / "results/step08_publication_audit"
OUT.mkdir(parents=True, exist_ok=True)

FIG = P / "results/step08_figures"
TAB = P / "results/step08_publication_tables"
PROV = P / "results/step08_provenance"
SRC = P / "results/step06_final_tables"

MANIFEST = OUT / "step08_17_publication_artifact_audit.json"
INVENTORY = OUT / "step08_17_artifact_inventory.csv"

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

groups = {
    "figure02": [
        FIG / "figure02_model_performance_95ci.png",
        FIG / "figure02_model_performance_95ci.pdf",
        FIG / "figure02_model_performance_95ci.json",
    ],
    "figure03": [
        FIG / "figure03_accuracy_stability_tradeoff.png",
        FIG / "figure03_accuracy_stability_tradeoff.pdf",
        FIG / "figure03_accuracy_stability_tradeoff.json",
    ],
    "figure04": [
        FIG / "figure04_lexical_variation_prediction_range.png",
        FIG / "figure04_lexical_variation_prediction_range.pdf",
        FIG / "figure04_lexical_variation_prediction_range.json",
    ],
    "figure05": [
        FIG / "figure05_critical_asr_hallucination.png",
        FIG / "figure05_critical_asr_hallucination.pdf",
        FIG / "figure05_critical_asr_hallucination_data.csv",
        FIG / "figure05_critical_asr_hallucination.json",
    ],
    "figure05b": [
        FIG / "figure05b_h2_h1_feature_contributions.png",
        FIG / "figure05b_h2_h1_feature_contributions.pdf",
        FIG / "figure05b_feature_contribution_summary.csv",
        FIG / "figure05b_feature_contribution_manifest.json",
    ],
    "publication_tables": [
        TAB / "table02a_model_performance.csv",
        TAB / "table02b_paired_bootstrap_comparisons.csv",
        TAB / "step08_15a_verified_key_contrasts.csv",
        TAB / "table03_anomaly_sensitivity_publication.csv",
        TAB / "step08_16_anomaly_sensitivity_manifest.json",
    ],
    "provenance": [
        PROV / "step08_07b_source_reconstruction.json",
        PROV / "step08_04_corrected_metadata_audit.json",
        PROV / "step08_08_asr_batch_integrity.json",
        PROV / "step08_09_final_provenance_record.json",
    ],
    "frozen_sources": [
        SRC / "table01_primary_models_with_ci.csv",
        SRC / "table02_paired_differences_with_ci.csv",
        SRC / "table03_anomaly_sensitivity.csv",
        SRC / "step06_asr_variability_per_audio.csv",
        SRC / "step06_c2_critical_h2_vs_h1_contributions.csv",
    ],
}

rows = []
missing = []

for group, paths in groups.items():
    for path in paths:
        exists = path.is_file()
        if not exists:
            missing.append(str(path.relative_to(P)))

        rows.append({
            "group": group,
            "path": str(path.relative_to(P)),
            "exists": exists,
            "size_bytes": path.stat().st_size if exists else 0,
            "sha256": sha(path) if exists else "",
        })

inventory = pd.DataFrame(rows)

# Independently check saved figure manifests against
# their actual PNG, PDF and CSV artifacts.
figure_manifests = {
    "figure02": (
        FIG / "figure02_model_performance_95ci.json",
        FIG / "figure02_model_performance_95ci.png",
        FIG / "figure02_model_performance_95ci.pdf",
        None
    ),
    "figure03": (
        FIG / "figure03_accuracy_stability_tradeoff.json",
        FIG / "figure03_accuracy_stability_tradeoff.png",
        FIG / "figure03_accuracy_stability_tradeoff.pdf",
        None
    ),
    "figure04": (
        FIG / "figure04_lexical_variation_prediction_range.json",
        FIG / "figure04_lexical_variation_prediction_range.png",
        FIG / "figure04_lexical_variation_prediction_range.pdf",
        None
    ),
    "figure05": (
        FIG / "figure05_critical_asr_hallucination.json",
        FIG / "figure05_critical_asr_hallucination.png",
        FIG / "figure05_critical_asr_hallucination.pdf",
        FIG / "figure05_critical_asr_hallucination_data.csv"
    ),
    "figure05b": (
        FIG / "figure05b_feature_contribution_manifest.json",
        FIG / "figure05b_h2_h1_feature_contributions.png",
        FIG / "figure05b_h2_h1_feature_contributions.pdf",
        FIG / "figure05b_feature_contribution_summary.csv"
    )
}

checks = []

for name, (manifest, png, pdf, csv) in figure_manifests.items():
    if not all(
        x.is_file()
        for x in [manifest, png, pdf]
        + ([csv] if csv is not None else [])
    ):
        checks.append({"figure": name, "status": "MISSING"})
        continue

    meta = json.loads(manifest.read_text(encoding="utf-8"))
    ok = (
        meta["png_sha256"] == sha(png)
        and meta["pdf_sha256"] == sha(pdf)
    )

    if csv is not None:
        key = (
            "data_csv_sha256"
            if name == "figure05"
            else "csv_sha256"
        )
        ok = ok and meta[key] == sha(csv)

    checks.append({
        "figure": name,
        "status": "PASS" if ok else "HASH_MISMATCH"
    })

checks_df = pd.DataFrame(checks)

# Verify the most important frozen statistical values.
models = pd.read_csv(
    SRC / "table01_primary_models_with_ci.csv"
)
paired = pd.read_csv(
    SRC / "table02_paired_differences_with_ci.csv"
)
sensitivity = pd.read_csv(
    SRC / "table03_anomaly_sensitivity.csv"
)

assert len(models) == 5
assert len(paired) == 18
assert len(sensitivity) == 15

assert set(models["model"]) == {"A","B1","B2","C1","C2"}
assert set(sensitivity["audio_count"]) == {400, 399, 396}

# Check the already verified provenance reports.
provenance = json.loads(
    (PROV / "step08_09_final_provenance_record.json")
    .read_text(encoding="utf-8")
)

assert provenance["status"] == "PROVENANCE_DOCUMENTED"

def safe_csv(df, path):
    tmp = path.with_suffix(".tmp")
    df.to_csv(tmp, index=False, encoding="utf-8")
    if path.exists():
        assert sha(path) == sha(tmp), (
            f"Existing inventory changed: {path}"
        )
        tmp.unlink()
    else:
        tmp.replace(path)

safe_csv(inventory, INVENTORY)

status = (
    "PASS"
    if len(missing) == 0
    and checks_df["status"].eq("PASS").all()
    else "NEEDS_REVIEW"
)

report = {
    "step": "8.17",
    "status": status,
    "files_expected": len(inventory),
    "files_present": int(inventory["exists"].sum()),
    "missing": missing,
    "figure_integrity_checks": checks,
    "model_rows": len(models),
    "paired_contrast_rows": len(paired),
    "sensitivity_rows": len(sensitivity),
    "provenance_status": provenance["status"],
    "figure01_workflow_created": False,
    "original_results_modified": False,
    "models_retrained": False,
    "inventory_sha256": sha(INVENTORY)
}

tmp = MANIFEST.with_suffix(".tmp")
tmp.write_text(
    json.dumps(report, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

if MANIFEST.exists():
    assert json.loads(
        MANIFEST.read_text(encoding="utf-8")
    ) == report, "Existing audit manifest differs"
    tmp.unlink()
else:
    tmp.replace(MANIFEST)

print("=== STEP 8.17 PUBLICATION ARTIFACT AUDIT ===")
print("Status:", status)
print("Files expected:", len(inventory))
print("Files present:", int(inventory["exists"].sum()))
print("Missing files:", len(missing))

print("\n=== FIGURE INTEGRITY ===")
print(checks_df.to_string(index=False))

print("\n=== FROZEN TABLE INTEGRITY ===")
print("Model rows:", len(models))
print("Paired contrast rows:", len(paired))
print("Sensitivity rows:", len(sensitivity))
print("Provenance:", provenance["status"])

print("\n=== REMAINING PUBLICATION WORK ===")
print("Figure 1 research workflow: NOT YET CREATED")
print("Captions and manuscript integration: NOT YET FINALIZED")

print("\n=== SAVED FILES ===")
print("Inventory:", INVENTORY)
print("Audit:", MANIFEST)
print("Audit SHA256:", sha(MANIFEST))
print("Original predictions modified: NO")


In [ ]:

from pathlib import Path
import json
import hashlib

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

P = Path("/kaggle/working/acoustic_linguistic_research")
OUT = P / "results/step08_figures"
OUT.mkdir(parents=True, exist_ok=True)

PNG = OUT / "figure01_research_workflow.png"
PDF = OUT / "figure01_research_workflow.pdf"
MANIFEST = OUT / "figure01_research_workflow.json"

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def immutable_save(path, writer):
    temp = path.with_name(path.stem + ".temporary" + path.suffix)
    writer(temp)
    if path.exists():
        assert sha(path) == sha(temp), (
            f"Existing artifact differs: {path}"
        )
        temp.unlink()
    else:
        temp.replace(path)

fig, ax = plt.subplots(figsize=(13.4, 10.0))
ax.set_xlim(0, 13.4)
ax.set_ylim(0, 10)
ax.axis("off")

def box(x, y, w, h, heading, details,
        face="#EDF3F7", edge="#61788A",
        fontsize=10):
    rect = FancyBboxPatch(
        (x, y), w, h,
        boxstyle="round,pad=0.12,rounding_size=0.16",
        linewidth=1.25,
        edgecolor=edge,
        facecolor=face
    )
    ax.add_patch(rect)
    ax.text(
        x + w/2, y + h*0.68, heading,
        ha="center", va="center",
        fontsize=fontsize + 1,
        fontweight="bold",
        color="#213343"
    )
    ax.text(
        x + w/2, y + h*0.32, details,
        ha="center", va="center",
        fontsize=fontsize - 0.6,
        color="#35485A",
        linespacing=1.28
    )

def arrow(x1, y1, x2, y2, color="#526A7B"):
    ax.add_patch(
        FancyArrowPatch(
            (x1, y1), (x2, y2),
            arrowstyle="-|>",
            mutation_scale=14,
            linewidth=1.5,
            color=color,
            connectionstyle="arc3,rad=0"
        )
    )

# 1. Source dataset.
box(
    3.8, 8.65, 5.8, 1.0,
    "SpeechOcean762 — Training Partition",
    "2,500 recordings / 125 speakers\n"
    "Official test partition reserved"
)
arrow(6.7, 8.55, 6.7, 8.13)

# 2. Pilot selection.
box(
    3.8, 7.1, 5.8, 0.95,
    "Speaker-Level Pilot Selection",
    "20 speakers × 20 recordings = 400 audios\n"
    "Recorded seed = 42"
)
arrow(6.7, 7.0, 6.7, 6.58)

# 3. Parallel extraction streams.
box(
    0.55, 5.3, 3.65, 1.12,
    "Acoustic Features",
    "14 audio-derived variables\n"
    "VAD, pauses, pitch, RMS",
    face="#E8F2ED",
    edge="#5B8876"
)

box(
    4.9, 5.3, 3.65, 1.12,
    "ASR Transcriptions",
    "Whisper small.en\n"
    "H1–H5 per recording",
    face="#E8EFF9",
    edge="#56799E"
)

box(
    9.2, 5.3, 3.65, 1.12,
    "Reading Prompt",
    "Reference text available\n"
    "in read-aloud task",
    face="#F8F0E3",
    edge="#A68954"
)

arrow(5.0, 7.0, 2.4, 6.52)
arrow(6.7, 7.0, 6.7, 6.52)
arrow(8.4, 7.0, 11.0, 6.52)

# 4. Feature group preparation.
box(
    0.55, 3.65, 3.65, 1.0,
    "A: Acoustic Features",
    "14 features / recording",
    face="#E8F2ED",
    edge="#5B8876"
)

box(
    4.9, 3.65, 3.65, 1.0,
    "B: Linguistic Features",
    "7 ASR-derived features\n"
    "per hypothesis",
    face="#E8EFF9",
    edge="#56799E"
)

box(
    9.2, 3.65, 3.65, 1.0,
    "Prompt-Aware Features",
    "2 prompt–ASR disagreement\n"
    "measures (not WER/CER)",
    face="#F8F0E3",
    edge="#A68954"
)

arrow(2.4, 5.2, 2.4, 4.78)
arrow(6.7, 5.2, 6.7, 4.78)
arrow(11.0, 5.2, 11.0, 4.78)
arrow(8.7, 5.85, 9.05, 4.23)

# 5. Model configurations.
box(
    0.6, 2.05, 12.2, 1.0,
    "Model Configurations",
    "M0: mean baseline   |   A: acoustic   |   B1: linguistic   |   "
    "C1: acoustic + linguistic\n"
    "Secondary: B2 = linguistic + prompt; C2 = full fusion + prompt",
    face="#F0F1F3",
    edge="#777F88",
    fontsize=9.8
)

arrow(2.4, 3.55, 2.4, 3.17)
arrow(6.7, 3.55, 6.7, 3.17)
arrow(11.0, 3.55, 11.0, 3.17)

# 6. Evaluation.
box(
    0.6, 0.45, 5.7, 0.95,
    "Five-Fold Speaker-Disjoint CV",
    "Ridge alpha=1.0; fold-only preprocessing\n"
    "All H1–H5 of each audio remain together",
    face="#EEF1F5",
    edge="#67798B",
    fontsize=9.5
)

box(
    7.05, 0.45, 5.75, 0.95,
    "Performance & Robustness Analysis",
    "MAE, RMSE, H1–H5 prediction range\n"
    "Speaker-cluster bootstrap + sensitivity",
    face="#EEF1F5",
    edge="#67798B",
    fontsize=9.5
)

arrow(6.7, 1.95, 3.45, 1.52)
arrow(6.3, 0.92, 6.92, 0.92)

fig.suptitle(
    "Research Workflow: ASR Variability and Human-Aligned Fluency Scoring",
    fontsize=16,
    fontweight="bold",
    y=0.995
)

fig.tight_layout(rect=[0.015, 0.015, 0.985, 0.965])

immutable_save(
    PNG,
    lambda path: fig.savefig(
        path, dpi=300, bbox_inches="tight"
    )
)
immutable_save(
    PDF,
    lambda path: fig.savefig(
        path,
        bbox_inches="tight",
        metadata={
            "Title": "Research Workflow",
            "Creator": "matplotlib",
            "CreationDate": None
        }
    )
)

plt.show()
plt.close(fig)

manifest = {
    "figure": "Figure 1",
    "status": "CREATED",
    "title": (
        "Research Workflow: ASR Variability and "
        "Human-Aligned Fluency Scoring"
    ),
    "dataset": "mispeech/speechocean762",
    "dataset_split": "train",
    "source_recordings": 2500,
    "source_speakers": 125,
    "selected_recordings": 400,
    "selected_speakers": 20,
    "asr_hypotheses_per_audio": 5,
    "feature_counts": {
        "acoustic": 14,
        "linguistic": 7,
        "prompt_aware": 2
    },
    "models": ["M0", "A", "B1", "B2", "C1", "C2"],
    "validation": "five-fold speaker-disjoint CV",
    "primary_alpha": 1.0,
    "primary_metrics": ["MAE", "RMSE"],
    "stability_metric": "prediction range across H1-H5",
    "bootstrap_clusters": "speaker",
    "interpretation_notes": [
        "Prompt-aware models require reading prompts.",
        "Prompt–ASR disagreement is not measured WER or CER.",
        "The official test partition was not used.",
        "Acoustic-only A is ASR-invariant by design.",
        "Figure illustrates the workflow, not causal inference."
    ],
    "png_sha256": sha(PNG),
    "pdf_sha256": sha(PDF)
}

immutable_save(
    MANIFEST,
    lambda path: path.write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
)

print("\n=== STEP 8.18 FIGURE 1 ===")
print("Status: PASS")
print("Dataset: SpeechOcean762 train")
print("Recordings: 400")
print("Speakers: 20")
print("ASR hypotheses: 2000")
print("Models: M0, A, B1, B2, C1, C2")

print("\n=== OUTPUT FILES ===")
print("PNG:", PNG)
print("PDF:", PDF)
print("Manifest:", MANIFEST)
print("Manifest SHA256:", sha(MANIFEST))
print("Original predictions modified: NO")
